<a href="https://colab.research.google.com/github/Joaoplims/NLP-HandsOn/blob/main/HOF02/HOF02_IntentionalityFilter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Download dados (brutos)

In [3]:
def convert_github_blob_to_raw(url: str) -> str:
    """
    Converts a GitHub blob URL to a raw usercontent URL.
    Example:
    From: https://github.com/user/repo/blob/main/file.csv
    To:   https://raw.githubusercontent.com/user/repo/main/file.csv
    """
    if "github.com" in url and "/blob/" in url:
        url = url.replace("github.com", "raw.githubusercontent.com")
        url = url.replace("/blob/", "/")
    return url

# Example usage:
test_url = "https://github.com/Joaoplims/NLP-HandsOn/blob/main/HOF02/spam.csv"
raw_url = convert_github_blob_to_raw(test_url)
print(f"Raw URL: {raw_url}")

Raw URL: https://raw.githubusercontent.com/Joaoplims/NLP-HandsOn/main/HOF02/spam.csv


In [4]:
import pandas as pd
from datasets import load_dataset
import kagglehub
import os
import glob

def process_url_and_create_df(dataset_item: dict) -> pd.DataFrame | None:
    """
    Loads data based on the source specified in the dataset_item dictionary.
    Supports: 'Github', 'Hugging Face', and 'Kaggle'.
    """
    source = dataset_item.get('source')
    address = dataset_item.get('address')

    try:
        if source == "Github":
            print(f"Downloading directly from GitHub: {address}")
            try:
                return pd.read_csv(address)
            except UnicodeDecodeError:
                print("UTF-8 decoding failed. Retrying with latin-1 encoding...")
                return pd.read_csv(address, encoding='latin-1')

        elif source == "Hugging Face":
            print(f"Loading Hugging Face dataset: {address}")
            ds = load_dataset(address)
            if hasattr(ds, 'keys'):
                split_name = 'train' if 'train' in ds else list(ds.keys())[0]
                return ds[split_name].to_pandas()
            return ds.to_pandas()

        elif source == "Kaggle":
            print(f"Downloading Kaggle dataset: {address}")
            path = kagglehub.dataset_download(address)
            print(f"Path to dataset files: {path}")

            # Search for CSV or JSONL files
            csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
            jsonl_files = glob.glob(os.path.join(path, "**", "*.jsonl"), recursive=True)

            if csv_files:
                print(f"Found CSV: {os.path.basename(csv_files[0])}")
                return pd.read_csv(csv_files[0])
            elif jsonl_files:
                print(f"Found JSONL: {os.path.basename(jsonl_files[0])}")
                return pd.read_json(jsonl_files[0], lines=True)
            else:
                all_files = glob.glob(os.path.join(path, "**", "*"), recursive=True)
                print(f"No supported files found. Files available: {[os.path.basename(f) for f in all_files if os.path.isfile(f)]}")
                return None

        else:
            print(f"Unsupported source: {source}")
            return None

    except Exception as e:
        print(f"Error loading from {source} ({address}): {e}")
        return None

In [5]:
import pandas as pd

# Updated list with structured sources
dataset_urls = [
    {"source": "Github", "address": "https://raw.githubusercontent.com/Joaoplims/NLP-HandsOn/main/HOF02/spam.csv"},
    {"source": "Github", "address": "https://media.githubusercontent.com/media/RockENZO/data/refs/heads/main/Synthetic-Data-for-Scam-Detection-Leveraging-LLMs-to-Train-Deep-Learning-Models-main/data/single-agent-scam-dialogue_train.csv"},
    {"source": "Github", "address": "https://media.githubusercontent.com/media/RockENZO/data/refs/heads/main/Synthetic-Data-for-Scam-Detection-Leveraging-LLMs-to-Train-Deep-Learning-Models-main/data/multi_agent_conversation_train.csv"},
    {"source": "Hugging Face", "address": "wangyuancheng/discord-phishing-scam-clean"},
    {"source": "Kaggle", "address": "kanishkaranaweera/synthetic-dialogue-dataset-romance-scam-detection"}
]

indexDataset = 2
item = dataset_urls[indexDataset]
scam_conversations_df = process_url_and_create_df(item)

if scam_conversations_df is not None:
    print(f"Successfully loaded dataset from {item['source']}:")
    display(scam_conversations_df.head())
else:
    print(f"Failed to load the dataset from {item['address']}.")

Successfully loaded dataset from Github:


,dialogue,personality,type,labels
0,"Innocent: Hello. Suspect: Hi, this is Karen f...",aggressive,appointment,0
1,"Innocent: Hello. Suspect: Hi, this is Karen f...",aggressive,appointment,0
2,"Innocent: Hello. Suspect: Hi, this is Karen f...",aggressive,appointment,0
3,"Innocent: Hello. Suspect: Hi, this is Rachel ...",aggressive,appointment,0
4,"Innocent: Hello. Suspect: Hi, this is Karen f...",aggressive,appointment,0


## EDA com os dados brutos


In [6]:
if scam_conversations_df is not None:
    print("\n--- EDA: Class Balance and Scam Type Distribution ---\n")
    # 1. Class Balance and Distribution of Scam Types
    if 'type' in scam_conversations_df.columns:
        type_counts = scam_conversations_df['type'].value_counts()
        print("Number of classes and their labels (Scam Type Distribution):")
        display(type_counts)

        most_common_type = type_counts.idxmax()
        print(f"\nThe most common scam type is: '{most_common_type}' with {type_counts.max()} occurrences.\n")
    else:
        print("The 'type' column was not found for class balance analysis.")

    print("\n--- EDA: Conversation Size Analysis ---\n")
    # 3. Analyze conversation size (number of words, characters, turns)
    if 'dialogue' in scam_conversations_df.columns:
        # Ensure 'dialogue' column is string type and handle NaNs
        scam_conversations_df['dialogue'] = scam_conversations_df['dialogue'].astype(str).fillna('')

        scam_conversations_df['num_words'] = scam_conversations_df['dialogue'].apply(lambda x: len(x.split()))
        scam_conversations_df['num_chars'] = scam_conversations_df['dialogue'].apply(lambda x: len(x))
        # Assuming each newline separates a turn. Add 1 for the first turn if not empty.
        scam_conversations_df['num_turns'] = scam_conversations_df['dialogue'].apply(lambda x: x.count('\n') + 1 if x else 0)

        print("Descriptive statistics for dialogue size by scam type:")
        if 'type' in scam_conversations_df.columns:
            conversation_size_analysis = scam_conversations_df.groupby('type')[['num_words', 'num_chars', 'num_turns']].agg(['mean', 'median', 'std', 'min', 'max'])
            display(conversation_size_analysis)
        else:
            conversation_size_analysis = scam_conversations_df[['num_words', 'num_chars', 'num_turns']].agg(['mean', 'median', 'std', 'min', 'max'])
            display(conversation_size_analysis)

    else:
        print("The 'dialogue' column was not found for conversation size analysis.")
else:
    print("The DataFrame 'scam_conversations_df' is empty or not loaded. Cannot perform EDA.")


--- EDA: Class Balance and Scam Type Distribution ---

Number of classes and their labels (Scam Type Distribution):


,count
type,
appointment,160
delivery,160
insurance,160
refund,160
reward,160
ssn,160
support,160
wrong,160



The most common scam type is: 'appointment' with 160 occurrences.


--- EDA: Conversation Size Analysis ---

Descriptive statistics for dialogue size by scam type:


num_words                                 num_chars          \
                  mean median         std  min   max        mean  median   
type                                                                       
appointment  290.36250  308.5  169.315770   79   934  1625.61875  1704.0   
delivery     362.81875  353.5  158.420262  106   963  2065.99375  2060.0   
insurance    691.13125  568.0  407.684304  149  1955  3922.85625  3266.0   
refund       657.39375  517.0  342.335230  220  1834  3750.61250  2937.0   
reward       618.48750  459.5  379.965043   69  2218  3504.76250  2620.5   
ssn          654.51250  513.5  322.395144  220  1712  3752.95000  2942.0   
support      775.95000  595.5  378.769736  205  1874  4394.20625  3451.0   
wrong        147.40000  130.0   49.422730   73   349   810.43125   727.0   

                                      num_turns                      
                     std   min    max      mean median  std min max  
type                                                                 
appointment   937.688110   460   5385       1.0    1.0  0.0   1   1  
delivery      890.658519   613   5365       1.0    1.0  0.0   1   1  
insurance    2313.214888   840  11449       1.0    1.0  0.0   1   1  
refund       1952.659634  1238  10779       1.0    1.0  0.0   1   1  
reward       2152.024741   436  13288       1.0    1.0  0.0   1   1  
ssn          1831.825566  1326   9901       1.0    1.0  0.0   1   1  
support      2105.629349  1199  10516       1.0    1.0  0.0   1   1  
wrong         266.005476   416   1856       1.0    1.0  0.0   1   1

## Text Processing

In [20]:
# Install necessary libraries
!pip install unidecode

# Download NLTK data
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')
# Ensure the necessary NLTK data is downloaded
nltk.download('averaged_perceptron_tagger')
# Download Scikit-Learn
!pip install -U scikit-learn

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [24]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tag import pos_tag

# Baixa os conjuntos de dados necessários se ainda não foram baixados
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger_eng')
except LookupError:
    nltk.download('averaged_perceptron_tagger_eng')

# Função auxiliar para mapear as POS tags do NLTK para as tags da WordNet
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN # Padrão para substantivo se não houver tag clara

def normalize_text(text):
    """Remove pontuação, stopwords, realiza POS-tagging e lematiza uma string de texto."""
    # 1. Remover pontuação
    translator = str.maketrans('', '', string.punctuation)
    text_no_punct = text.translate(translator)

    # 2. Converter para minúsculas e tokenizar
    words = text_no_punct.lower().split()

    # 3. Remover stopwords
    stop_words = set(stopwords.words('english'))
    filtered_words = [word for word in words if word not in stop_words]

    # 4. POS-tagging (Parte da Fala)
    pos_tagged_words = pos_tag(filtered_words)

    # 5. Lematização
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = []
    for word, tag in pos_tagged_words:
        wntag = get_wordnet_pos(tag)
        lemmatized_words.append(lemmatizer.lemmatize(word, wntag))

    return ' '.join(lemmatized_words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [25]:
# Cada mensagem está tokenizada em seu respectivo vetor
normalized_msg = scam_conversations_df["dialogue"].apply(normalize_text)
print(scam_conversations_df["dialogue"][0])
print(normalized_msg[0])

Innocent: Hello.  Suspect: Hi, this is Karen from Smith & Co. I'm calling to confirm your appointment with Dr. Lee on Thursday at 2 PM. Can you please confirm that you're still available to come in at that time? Innocent: What's the point of confirming? You people always change things at the last minute anyway. What's the real reason you're calling? Suspect: I understand your concern, Mr. Johnson. I apologize if we've had any last-minute changes in the past. However, I assure you that this call is just a routine confirmation to ensure you're still able to make the appointment. We want to make sure we're both on the same page and avoid any misunderstandings. Can you please confirm your availability for Thursday at 2 PM? Innocent: Save it, Karen. I don't need some scripted apology from a customer service drone. You're just trying to cover your behind. What's the doctor's schedule really looking like? Is he running behind again? Suspect: I understand your frustration, Mr. Johnson. To be h

## Representação Vetorial (TODO)